# Churn-risk retention campaign: auditable end-to-end solution

This notebook performs the complete reproducible process: it isolates a stratified 20% customer list, uses only the remaining 80% to compare probability models and evaluate the intervention strategy, then applies the frozen CatBoost + dynamic-programming policy once to the held-out list.

The campaign starts with **S/1,000**. Each intervention costs **S/10**. A revealed `Churn=Yes` outcome adds **S/100**. Final profit is final budget minus S/1,000.


## Controls and data handling

All randomized operations use `SEED=42`. The split is fixed at 80% training and 20% test with stratification on churn. `customerID` is traceability-only and never enters a predictive model. `TotalCharges` blank strings are converted to numeric missing values before splitting; learned preprocessing for Logistic Regression and XGBoost is fitted within each training fold. CatBoost handles the validated categorical columns and numeric missing values natively.

The test customers are not used below until the final evaluation section. Model selection, probability diagnostics, ranking, and policy comparison use training-only out-of-fold (OOF) predictions.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

SEED = 42
TEST_SIZE = 0.20
INITIAL_BUDGET, COST, REWARD = 1000, 10, 100
FILENAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"


def locate_csv(start=Path.cwd()):
    for folder in (start, *start.parents):
        csv_path = folder / FILENAME
        if csv_path.is_file():
            return csv_path
    raise FileNotFoundError(
        f"Place {FILENAME} beside this notebook or in a parent folder."
    )


data = pd.read_csv(locate_csv())
if data["customerID"].duplicated().any():
    raise ValueError("customerID must be unique.")
blank_total_charges = data["TotalCharges"].astype(str).str.strip().eq("").sum()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1}).astype(int)

ids, y = data["customerID"].copy(), data["Churn"].copy()
X = data.drop(columns=["customerID", "Churn"])
NUMERIC = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL = [column for column in X.columns if column not in NUMERIC]
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

controls = pd.DataFrame(
    [
        ["Random seed", SEED],
        ["Train / test split", "80% / 20%, stratified on Churn"],
        ["Training customers", len(X_train)],
        ["Held-out customers", len(X_test)],
        ["Initial budget / cost / reward", "S/1,000 / S/10 / S/100"],
        ["Customer ID predictive feature", "No"],
        ["Blank TotalCharges converted", blank_total_charges],
    ],
    columns=["Control", "Value"],
)
display(controls)
display(
    pd.DataFrame(
        [
            {
                "rows": len(data),
                "churn_prevalence": data["Churn"].mean(),
                "missing_values_after_TotalCharges_conversion": int(
                    data.isna().sum().sum()
                ),
            }
        ]
    )
)

,Control,Value
0,Random seed,42
1,Train / test split,"80% / 20%, stratified on Churn"
2,Training customers,5634
3,Held-out customers,1409
4,Initial budget / cost / reward,"S/1,000 / S/10 / S/100"
5,Customer ID predictive feature,No
6,Blank TotalCharges converted,11


,rows,churn_prevalence,missing_values_after_TotalCharges_conversion
0,7043,0.26537,11


## Decision policy

The predictive model supplies a churn probability for every customer. Customers are ranked by descending probability, with ascending `customerID` as the deterministic tie-break. This is a risk-priority ranking; the dynamic program is exact **conditional on this ranking**, not a claim of globally optimal joint ordering across every permutation.

For queue position `i` and current budget `B`, stopping is worth `B`. Continuing is feasible only when `B ≥ 10` and has value:

`p_i × V(i+1, B + 90) + (1 − p_i) × V(i+1, B − 10)`

The recurrence chooses the larger of stopping and continuing. The implementation stores balances in S/10 units, so every state transition is exact. There is no capacity fraction, fixed Top-K count, or tuned probability threshold.


In [2]:
def build_dp_policy(probabilities, customer_ids, risk_order=True):
    queue_ids = np.asarray(pd.Series(customer_ids).astype(str))
    queue_p = np.asarray(probabilities, dtype=float)
    if risk_order:
        order = np.lexsort((queue_ids, -queue_p))
        queue_ids, queue_p = queue_ids[order], queue_p[order]
    n, start_units, reward_units = len(queue_p), INITIAL_BUDGET // COST, REWARD // COST
    actions = [None] * n
    next_values = start_units - n + reward_units * np.arange(n + 1, dtype=float)
    for i in range(n - 1, -1, -1):
        successes = np.arange(i + 1)
        balance = start_units - i + reward_units * successes
        continue_value = (
            queue_p[i] * next_values[1:] + (1 - queue_p[i]) * next_values[:-1]
        )
        continue_action = (balance >= 1) & (continue_value > balance)
        values = balance.astype(float)
        values[continue_action] = continue_value[continue_action]
        actions[i], next_values = continue_action, values
    return {
        "ids": queue_ids,
        "p": queue_p,
        "actions": actions,
        "expected_final_budget": float(next_values[0] * COST),
    }


def simulate_dp(policy, customer_ids, labels):
    labels_by_id = pd.Series(
        np.asarray(labels, dtype=int),
        index=pd.Series(customer_ids).astype(str).to_numpy(),
    )
    balance, successes, attempted, reason = INITIAL_BUDGET, 0, 0, "queue_complete"
    for i, customer_id in enumerate(policy["ids"]):
        if balance < COST:
            reason = "insufficient_budget"
            break
        if not policy["actions"][i][successes]:
            reason = "voluntary_stop"
            break
        balance = balance - COST + REWARD * int(labels_by_id[customer_id])
        successes += int(labels_by_id[customer_id])
        attempted += 1
    return {
        "expected_final_budget": policy["expected_final_budget"],
        "expected_profit": policy["expected_final_budget"] - INITIAL_BUDGET,
        "final_budget": balance,
        "realized_profit": balance - INITIAL_BUDGET,
        "attempted_interventions": attempted,
        "successful_interventions": successes,
        "failed_interventions": attempted - successes,
        "termination_reason": reason,
    }


def threshold_campaign(probabilities, customer_ids, labels):
    queue_ids, queue_p = np.asarray(pd.Series(customer_ids).astype(str)), np.asarray(
        probabilities
    )
    order = np.lexsort((queue_ids, -queue_p))
    queue_ids, queue_p = queue_ids[order], queue_p[order]
    labels_by_id = pd.Series(
        np.asarray(labels, dtype=int),
        index=pd.Series(customer_ids).astype(str).to_numpy(),
    )
    balance = INITIAL_BUDGET
    successes = attempted = 0
    for customer_id, probability in zip(queue_ids, queue_p):
        if balance < COST or probability <= COST / REWARD:
            break
        churned = int(labels_by_id[customer_id])
        balance += -COST + REWARD * churned
        successes += churned
        attempted += 1
    return {
        "final_budget": balance,
        "realized_profit": balance - INITIAL_BUDGET,
        "attempted_interventions": attempted,
        "successful_interventions": successes,
    }

## Training-only model comparison and strategy evidence

Logistic Regression provides a simple calibrated baseline. XGBoost and CatBoost test nonlinear probability models. All preprocessing that learns from data is inside the relevant training fold. Models are ranked by realized OOF campaign profit from the DP policy; expected OOF profit, log loss, and simplicity break ties deterministically.

The strategy comparison uses the selected model's same OOF probabilities. The `p > 0.10` rule is the single-customer break-even baseline because `100p − 10 > 0`. DP is retained as the exact stopping policy for the stated queue and economics.


In [3]:
def preprocessor(scale_numeric):
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    return ColumnTransformer(
        [
            ("numeric", Pipeline(numeric_steps), NUMERIC),
            (
                "categorical",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                CATEGORICAL,
            ),
        ]
    )


def make_model(name):
    if name == "logistic_regression":
        return Pipeline(
            [
                ("preprocess", preprocessor(True)),
                ("model", LogisticRegression(C=10.0, max_iter=2000, random_state=SEED)),
            ]
        )
    if name == "xgboost":
        return Pipeline(
            [
                ("preprocess", preprocessor(False)),
                (
                    "model",
                    XGBClassifier(
                        n_estimators=500,
                        learning_rate=0.03,
                        max_depth=3,
                        subsample=0.85,
                        colsample_bytree=0.85,
                        reg_lambda=1.0,
                        objective="binary:logistic",
                        eval_metric="logloss",
                        n_jobs=-1,
                        random_state=SEED,
                    ),
                ),
            ]
        )
    if name == "catboost":
        return CatBoostClassifier(
            iterations=200,
            depth=5,
            learning_rate=0.05,
            l2_leaf_reg=3.0,
            loss_function="Logloss",
            random_seed=SEED,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
        )
    raise KeyError(name)


def fit_predict(name, X_fit, y_fit, X_score):
    fitted = make_model(name)
    if name == "catboost":
        fitted.fit(X_fit, y_fit, cat_features=CATEGORICAL)
    else:
        fitted.fit(X_fit, y_fit)
    return fitted, fitted.predict_proba(X_score)[:, 1]


def oof_predict(name):
    probability = np.zeros(len(X_train))
    for fit_index, valid_index in StratifiedKFold(
        5, shuffle=True, random_state=SEED
    ).split(X_train, y_train):
        _, probability[valid_index] = fit_predict(
            name,
            X_train.iloc[fit_index],
            y_train.iloc[fit_index],
            X_train.iloc[valid_index],
        )
    return probability


candidate_names = ["logistic_regression", "xgboost", "catboost"]
candidate_probability = {name: oof_predict(name) for name in candidate_names}
complexity = {"logistic_regression": 0, "xgboost": 1, "catboost": 2}
rows = []
for name, probability in candidate_probability.items():
    campaign = simulate_dp(build_dp_policy(probability, id_train), id_train, y_train)
    rows.append(
        {
            "model": name,
            "ROC-AUC": roc_auc_score(y_train, probability),
            "PR-AUC": average_precision_score(y_train, probability),
            "Log loss": log_loss(y_train, probability),
            "Brier score": brier_score_loss(y_train, probability),
            "Expected OOF profit": campaign["expected_profit"],
            "Realized OOF profit": campaign["realized_profit"],
            "complexity": complexity[name],
        }
    )
model_evidence = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "Realized OOF profit",
            "Expected OOF profit",
            "Log loss",
            "complexity",
            "model",
        ],
        ascending=[False, False, True, True, True],
    )
    .reset_index(drop=True)
)
selected_model = model_evidence.loc[0, "model"]
display(model_evidence.drop(columns="complexity").round(4))

selected_oof = candidate_probability[selected_model]
dp_oof = simulate_dp(build_dp_policy(selected_oof, id_train), id_train, y_train)
random_order = np.random.default_rng(SEED).permutation(len(id_train))
random_oof = simulate_dp(
    build_dp_policy(
        selected_oof[random_order], id_train.iloc[random_order], risk_order=False
    ),
    id_train,
    y_train,
)
threshold_oof = threshold_campaign(selected_oof, id_train, y_train)
strategy_evidence = pd.DataFrame(
    [
        {
            "strategy": "no intervention",
            "realized_OOF_profit": 0,
            "final_budget": INITIAL_BUDGET,
            "interventions": 0,
        },
        {
            "strategy": "seeded random queue + DP",
            "realized_OOF_profit": random_oof["realized_profit"],
            "final_budget": random_oof["final_budget"],
            "interventions": random_oof["attempted_interventions"],
        },
        {
            "strategy": "risk queue + p > 0.10",
            "realized_OOF_profit": threshold_oof["realized_profit"],
            "final_budget": threshold_oof["final_budget"],
            "interventions": threshold_oof["attempted_interventions"],
        },
        {
            "strategy": "risk queue + DP",
            "realized_OOF_profit": dp_oof["realized_profit"],
            "final_budget": dp_oof["final_budget"],
            "interventions": dp_oof["attempted_interventions"],
        },
    ]
)
display(strategy_evidence)
print(
    f"Frozen selection: {selected_model} + descending-risk queue + exact DP stopping."
)

,model,ROC-AUC,PR-AUC,Log loss,Brier score,Expected OOF profit,Realized OOF profit
0,catboost,0.8490,0.6668,0.4114,0.1334,106025.0749,106020
1,logistic_regression,0.8458,0.6578,0.4165,0.1350,107262.1677,105410
2,xgboost,0.8474,0.6652,0.4136,0.1340,107447.6726,104620


,strategy,realized_OOF_profit,final_budget,interventions
0,no intervention,0,1000,0
1,seeded random queue + DP,93170,94170,5633
2,risk queue + p > 0.10,106020,107020,3528
3,risk queue + DP,106020,107020,3528


Frozen selection: catboost + descending-risk queue + exact DP stopping.


## Final held-out customer list

The selection above is complete. CatBoost is now fit once on all 80% training customers. The resulting probabilities order the held-out 20% customer list, and the same pre-specified DP policy processes it sequentially. Test outcomes are revealed only after an intervention to calculate the final balance.


In [4]:
final_model, test_probability = fit_predict(selected_model, X_train, y_train, X_test)
test_campaign = simulate_dp(build_dp_policy(test_probability, id_test), id_test, y_test)
final_result = pd.DataFrame(
    [
        ["Selected model", selected_model],
        ["Strategy", "Descending churn probability + exact DP stopping"],
        [
            "Interventions / successes",
            f"{test_campaign['attempted_interventions']} / {test_campaign['successful_interventions']}",
        ],
        ["Final budget", f"S/{test_campaign['final_budget']:,.2f}"],
        ["Final profit", f"S/{test_campaign['realized_profit']:,.2f}"],
    ],
    columns=["Measure", "Result"],
)
display(final_result)

,Measure,Result
0,Selected model,catboost
1,Strategy,Descending churn probability + exact DP stopping
2,Interventions / successes,874 / 353
3,Final budget,"S/27,560.00"
4,Final profit,"S/26,560.00"
